
## Respiratory Sound Classification with Pretrained End-to-End Deep Learning Approaches

Master's thesis project exploring deep learning approaches to respiratory sound classification using the ICBHI respiratory sound dataset.


### Overview

This project investigates the classification of respiratory-cycle audio as **Normal** or **Abnormal**, using pretrained audio models.



### How To Start
#### Dataset
1. Dataset must be downloaded. Go to the followig link and download the dataset from the official website:
[ICBHI_2017_Challenge](https://bhichallenge.med.auth.gr/ICBHI_2017_Challenge)

2. Create a new folder inside the `ICBHI_dataset/` directory, and name it "audios".

3. Copy only all 920 audio files (_.wav) from downloaded dataset, and paste them in the "audios" folder, `ICBHI_dataset/audios/`.

#### Extracting Audio at Cycle Level
run all the following cells, then every thing is set and we are good to go to start with the training  and modeling files in the root directory.

In [11]:
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm
from pathlib import Path

In [2]:
df = pd.read_csv("metadata_df.csv")

In [5]:
audio_cycles_df = df[['start', 'end', 'source_file']]

In [6]:
audio_cycles_df.head()

,start,end,source_file
0,0.036,0.579,101_1b1_Al_sc_Meditron
1,0.579,2.450,101_1b1_Al_sc_Meditron
2,2.450,3.893,101_1b1_Al_sc_Meditron
3,3.893,5.793,101_1b1_Al_sc_Meditron
4,5.793,7.521,101_1b1_Al_sc_Meditron


In [8]:
# Directories
input_dir = Path("audios/")
output_dir = Path("audios_cycles")
output_dir.mkdir(parents=True, exist_ok=True)

In [13]:
# Track progress over all audio cycles
for idx, row in tqdm(
    audio_cycles_df.iterrows(),
    total=len(audio_cycles_df),
    desc="Splitting audio cycles",
    unit="cycle"
):
    audio_name = str(row["source_file"])

    if not audio_name.endswith(".wav"):
        audio_name += ".wav"

    audio_path = input_dir / audio_name

    if not audio_path.exists():
        tqdm.write(f"File not found: {audio_path}")
        continue

    # Load audio while preserving the original sample rate
    y, sr = librosa.load(audio_path, sr=None, mono=False)

    # Convert timestamps to sample indices
    start_sample = max(0, int(row["start"] * sr))
    end_sample = min(y.shape[-1], int(row["end"] * sr))

    if end_sample <= start_sample:
        tqdm.write(f"Invalid segment: {audio_name}, row {idx}")
        continue

    # Extract the mini-audio
    mini_audio = y[..., start_sample:end_sample]

    # Convert channels-first to channels-last for soundfile
    if mini_audio.ndim == 2:
        mini_audio = mini_audio.T

    # Save with a unique filename
    output_name = f"{Path(audio_name).stem}_{row["start"]}_{row["end"]}.wav"
    output_path = output_dir / output_name

    sf.write(output_path, mini_audio, sr)

print("Audio splitting completed!")

Splitting audio cycles: 100%|███████████████████████████████████████████████████| 6898/6898 [02:41<00:00, 42.64cycle/s]

Audio splitting completed!


### Next Step: 
Preprocessing and audio cycles extraction step is done. <br>
To start workinng on the training and modeling by MobileNets, start the following file in the root directory:<br> 
- `ex_icbhi.py`